# Fraud-model sensitivity to transaction amount changes

Author: Muhammad Salman Shamsi

This public workflow consolidates the supplied learning notebook. It uses simulated transactions only. Local machine paths, diagnostic directory listings and saved cell outputs have been removed.

Simulation functions below derive from *Reproducible Machine Learning for Credit Card Fraud Detection – Practical Handbook* by Yann-Aël Le Borgne, Wissam Siblini, Bertrand Lebichot and Gianluca Bontempi (2022). Original code: https://github.com/Fraud-Detection-Handbook/fraud-detection-handbook. Notebook code license: GNU GPL v3. See LICENSE and ATTRIBUTION.md.

The amount experiments are individual counterfactuals with frozen historical features, not a sequential replay. This is an exploratory study. The original notebook used test comparisons to choose the calibration method and reused August data for calibration and threshold selection. This consolidated workflow fixes isotonic calibration and threshold 0.145 to reproduce that study; doing so does not erase those original selection limitations.


In [ ]:
from pathlib import Path
import random
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent
raw_folder = project_root / "data" / "simulated-data-raw"
results_folder = project_root / "results" / "reproduced"
results_folder.mkdir(parents=True, exist_ok=True)
print("Project folder:", project_root)


## Synthetic transaction generator

The following six functions are retained from the supplied notebook and attributed to the Fraud Detection Handbook. Generated data uses April–September 2018, 5,000 customer profiles and 10,000 terminal profiles.

In [ ]:
def generate_customer_profiles_table(n_customers, random_state=0):
    
    np.random.seed(random_state)
        
    customer_id_properties=[]
    
    # Generate customer properties from random distributions 
    for customer_id in range(n_customers):
        
        x_customer_id = np.random.uniform(0,100)
        y_customer_id = np.random.uniform(0,100)
        
        mean_amount = np.random.uniform(5,100) # Arbitrary (but sensible) value 
        std_amount = mean_amount/2 # Arbitrary (but sensible) value
        
        mean_nb_tx_per_day = np.random.uniform(0,4) # Arbitrary (but sensible) value 
        
        customer_id_properties.append([customer_id,
                                      x_customer_id, y_customer_id,
                                      mean_amount, std_amount,
                                      mean_nb_tx_per_day])
        
    customer_profiles_table = pd.DataFrame(customer_id_properties, columns=['CUSTOMER_ID',
                                                                      'x_customer_id', 'y_customer_id',
                                                                      'mean_amount', 'std_amount',
                                                                      'mean_nb_tx_per_day'])
    
    return customer_profiles_table

In [ ]:
def generate_terminal_profiles_table(n_terminals, random_state=0):
    
    np.random.seed(random_state)
        
    terminal_id_properties=[]
    
    # Generate terminal properties from random distributions 
    for terminal_id in range(n_terminals):
        
        x_terminal_id = np.random.uniform(0,100)
        y_terminal_id = np.random.uniform(0,100)
        
        terminal_id_properties.append([terminal_id,
                                      x_terminal_id, y_terminal_id])
                                       
    terminal_profiles_table = pd.DataFrame(terminal_id_properties, columns=['TERMINAL_ID',
                                                                      'x_terminal_id', 'y_terminal_id'])
    
    return terminal_profiles_table

In [ ]:
def get_list_terminals_within_radius(customer_profile, x_y_terminals, r):
    
    # Use numpy arrays in the following to speed up computations
    
    # Location (x,y) of customer as numpy array
    x_y_customer = customer_profile[['x_customer_id','y_customer_id']].values.astype(float)
    
    # Squared difference in coordinates between customer and terminal locations
    squared_diff_x_y = np.square(x_y_customer - x_y_terminals)
    
    # Sum along rows and compute suared root to get distance
    dist_x_y = np.sqrt(np.sum(squared_diff_x_y, axis=1))
    
    # Get the indices of terminals which are at a distance less than r
    available_terminals = list(np.where(dist_x_y<r)[0])
    
    # Return the list of terminal IDs
    return available_terminals
    

In [ ]:
def generate_transactions_table(customer_profile, start_date = "2018-04-01", nb_days = 10):
    
    customer_transactions = []
    
    random.seed(int(customer_profile.CUSTOMER_ID))
    np.random.seed(int(customer_profile.CUSTOMER_ID))
    
    # For all days
    for day in range(nb_days):
        
        # Random number of transactions for that day 
        nb_tx = np.random.poisson(customer_profile.mean_nb_tx_per_day)
        
        # If nb_tx positive, let us generate transactions
        if nb_tx>0:
            
            for tx in range(nb_tx):
                
                # Time of transaction: Around noon, std 20000 seconds. This choice aims at simulating the fact that 
                # most transactions occur during the day.
                time_tx = int(np.random.normal(86400/2, 20000))
                
                # If transaction time between 0 and 86400, let us keep it, otherwise, let us discard it
                if (time_tx>0) and (time_tx<86400):
                    
                    # Amount is drawn from a normal distribution  
                    amount = np.random.normal(customer_profile.mean_amount, customer_profile.std_amount)
                    
                    # If amount negative, draw from a uniform distribution
                    if amount<0:
                        amount = np.random.uniform(0,customer_profile.mean_amount*2)
                    
                    amount=np.round(amount,decimals=2)
                    
                    if len(customer_profile.available_terminals)>0:
                        
                        terminal_id = random.choice(customer_profile.available_terminals)
                    
                        customer_transactions.append([time_tx+day*86400, day,
                                                      customer_profile.CUSTOMER_ID, 
                                                      terminal_id, amount])
            
    customer_transactions = pd.DataFrame(customer_transactions, columns=['TX_TIME_SECONDS', 'TX_TIME_DAYS', 'CUSTOMER_ID', 'TERMINAL_ID', 'TX_AMOUNT'])
    
    if len(customer_transactions)>0:
        customer_transactions['TX_DATETIME'] = pd.to_datetime(customer_transactions["TX_TIME_SECONDS"], unit='s', origin=start_date)
        customer_transactions=customer_transactions[['TX_DATETIME','CUSTOMER_ID', 'TERMINAL_ID', 'TX_AMOUNT','TX_TIME_SECONDS', 'TX_TIME_DAYS']]
    
    return customer_transactions  
    
    

In [ ]:
def generate_dataset(n_customers = 10000, n_terminals = 1000000, nb_days=90, start_date="2018-04-01", r=5):
    
    start_time=time.time()
    customer_profiles_table = generate_customer_profiles_table(n_customers, random_state = 0)
    print("Time to generate customer profiles table: {0:.2}s".format(time.time()-start_time))
    
    start_time=time.time()
    terminal_profiles_table = generate_terminal_profiles_table(n_terminals, random_state = 1)
    print("Time to generate terminal profiles table: {0:.2}s".format(time.time()-start_time))
    
    start_time=time.time()
    x_y_terminals = terminal_profiles_table[['x_terminal_id','y_terminal_id']].values.astype(float)
    customer_profiles_table['available_terminals'] = customer_profiles_table.apply(lambda x : get_list_terminals_within_radius(x, x_y_terminals=x_y_terminals, r=r), axis=1)
    # With Pandarallel
    #customer_profiles_table['available_terminals'] = customer_profiles_table.parallel_apply(lambda x : get_list_closest_terminals(x, x_y_terminals=x_y_terminals, r=r), axis=1)
    customer_profiles_table['nb_terminals']=customer_profiles_table.available_terminals.apply(len)
    print("Time to associate terminals to customers: {0:.2}s".format(time.time()-start_time))
    
    start_time=time.time()
    transactions_df=customer_profiles_table.groupby('CUSTOMER_ID').apply(lambda x : generate_transactions_table(x.iloc[0], nb_days=nb_days)).reset_index(drop=True)
    # With Pandarallel
    #transactions_df=customer_profiles_table.groupby('CUSTOMER_ID').parallel_apply(lambda x : generate_transactions_table(x.iloc[0], nb_days=nb_days)).reset_index(drop=True)
    print("Time to generate transactions: {0:.2}s".format(time.time()-start_time))
    
    # Sort transactions chronologically
    transactions_df=transactions_df.sort_values('TX_DATETIME')
    # Reset indices, starting from 0
    transactions_df.reset_index(inplace=True,drop=True)
    transactions_df.reset_index(inplace=True)
    # TRANSACTION_ID are the dataframe indices, starting from 0
    transactions_df.rename(columns = {'index':'TRANSACTION_ID'}, inplace = True)
    
    return (customer_profiles_table, terminal_profiles_table, transactions_df)
    

In [ ]:
def add_frauds(customer_profiles_table, terminal_profiles_table, transactions_df):
    
    # By default, all transactions are genuine
    transactions_df['TX_FRAUD']=0
    transactions_df['TX_FRAUD_SCENARIO']=0
    
    # Scenario 1
    transactions_df.loc[transactions_df.TX_AMOUNT>220, 'TX_FRAUD']=1
    transactions_df.loc[transactions_df.TX_AMOUNT>220, 'TX_FRAUD_SCENARIO']=1
    nb_frauds_scenario_1=transactions_df.TX_FRAUD.sum()
    print("Number of frauds from scenario 1: "+str(nb_frauds_scenario_1))
    
    # Scenario 2
    for day in range(transactions_df.TX_TIME_DAYS.max()):
        
        compromised_terminals = terminal_profiles_table.TERMINAL_ID.sample(n=2, random_state=day)
        
        compromised_transactions=transactions_df[(transactions_df.TX_TIME_DAYS>=day) & 
                                                    (transactions_df.TX_TIME_DAYS<day+28) & 
                                                    (transactions_df.TERMINAL_ID.isin(compromised_terminals))]
                            
        transactions_df.loc[compromised_transactions.index,'TX_FRAUD']=1
        transactions_df.loc[compromised_transactions.index,'TX_FRAUD_SCENARIO']=2
    
    nb_frauds_scenario_2=transactions_df.TX_FRAUD.sum()-nb_frauds_scenario_1
    print("Number of frauds from scenario 2: "+str(nb_frauds_scenario_2))
    
    # Scenario 3
    for day in range(transactions_df.TX_TIME_DAYS.max()):
        
        compromised_customers = customer_profiles_table.CUSTOMER_ID.sample(n=3, random_state=day).values
        
        compromised_transactions=transactions_df[(transactions_df.TX_TIME_DAYS>=day) & 
                                                    (transactions_df.TX_TIME_DAYS<day+14) & 
                                                    (transactions_df.CUSTOMER_ID.isin(compromised_customers))]
        
        nb_compromised_transactions=len(compromised_transactions)
        
        
        random.seed(day)
        index_fauds = random.sample(list(compromised_transactions.index.values),k=int(nb_compromised_transactions/3))
        
        transactions_df.loc[index_fauds,'TX_AMOUNT']=transactions_df.loc[index_fauds,'TX_AMOUNT']*5
        transactions_df.loc[index_fauds,'TX_FRAUD']=1
        transactions_df.loc[index_fauds,'TX_FRAUD_SCENARIO']=3
        
                             
    nb_frauds_scenario_3=transactions_df.TX_FRAUD.sum()-nb_frauds_scenario_2-nb_frauds_scenario_1
    print("Number of frauds from scenario 3: "+str(nb_frauds_scenario_3))
    
    return transactions_df                 

In [ ]:
expected_dates = pd.date_range("2018-04-01", "2018-09-30")
expected_files = [raw_folder / f"{d:%Y-%m-%d}.pkl" for d in expected_dates]

# Load local simulation files if complete; otherwise regenerate in memory.
# The dataset is not included in this repository.
if all(file.exists() for file in expected_files):
    transactions_df = pd.concat(
        [pd.read_pickle(file) for file in expected_files], ignore_index=True
    )
    print("Loaded existing local simulation files.")
else:
    customer_profiles_table, terminal_profiles_table, transactions_df = generate_dataset(
        n_customers=5000, n_terminals=10000, nb_days=183,
        start_date="2018-04-01", r=5
    )
    transactions_df = add_frauds(
        customer_profiles_table, terminal_profiles_table, transactions_df
    )
    print("Generated synthetic data in memory.")

assert len(transactions_df) == 1754155, "Dataset differs from the original study."
assert int(transactions_df["TX_FRAUD"].sum()) == 14681
print("Transactions:", len(transactions_df))


## Historical features and chronological split

Sort rows chronologically before computing history. Each feature excludes the current row. Same-time transactions use the retained row order, which assumes sequential processing. Training covers April–July, validation August and testing September.

In [ ]:
transactions_df = transactions_df.sort_values(
    "TX_DATETIME",
    kind="mergesort"
).reset_index(drop=True)


In [ ]:
customer_group = transactions_df.groupby(
    "CUSTOMER_ID",
    sort=False
)

transactions_df["CUSTOMER_TX_COUNT_PREV"] = (
    customer_group.cumcount()
)

In [ ]:
customer_amount_sum = (
    customer_group["TX_AMOUNT"].cumsum()
    - transactions_df["TX_AMOUNT"]
)

transactions_df["CUSTOMER_AVG_AMOUNT_PREV"] = (
    customer_amount_sum
    / transactions_df["CUSTOMER_TX_COUNT_PREV"]
)

In [ ]:
transactions_df["CUSTOMER_AVG_AMOUNT_PREV"] = (
    transactions_df["CUSTOMER_AVG_AMOUNT_PREV"].fillna(0)
)

In [ ]:
transactions_df["CUSTOMER_AMOUNT_DIFF"] = (
    transactions_df["TX_AMOUNT"]
    - transactions_df["CUSTOMER_AVG_AMOUNT_PREV"]
)

In [ ]:
terminal_group = transactions_df.groupby(
    "TERMINAL_ID",
    sort=False
)

transactions_df["TERMINAL_TX_COUNT_PREV"] = (
    terminal_group.cumcount()
)

In [ ]:
transactions_df["TX_HOUR"] = (
    transactions_df["TX_DATETIME"].dt.hour
)

transactions_df["TX_DAY_OF_WEEK_NUM"] = (
    transactions_df["TX_DATETIME"].dt.dayofweek
)

transactions_df["TX_MONTH"] = (
    transactions_df["TX_DATETIME"].dt.month
)

In [ ]:
train_data = transactions_df[
    transactions_df["TX_DATETIME"] < "2018-08-01"
].copy()

validation_data = transactions_df[
    (transactions_df["TX_DATETIME"] >= "2018-08-01")
    & (transactions_df["TX_DATETIME"] < "2018-09-01")
].copy()

test_data = transactions_df[
    transactions_df["TX_DATETIME"] >= "2018-09-01"
].copy()

In [ ]:
behavioral_features = [
    "TX_AMOUNT", "TX_HOUR", "TX_DAY_OF_WEEK_NUM", "TX_MONTH",
    "CUSTOMER_TX_COUNT_PREV", "CUSTOMER_AMOUNT_DIFF", "TERMINAL_TX_COUNT_PREV"
]
X_train_behavioral = train_data[behavioral_features].copy()
X_validation_behavioral = validation_data[behavioral_features].copy()
X_test_behavioral = test_data[behavioral_features].copy()
y_train = train_data["TX_FRAUD"]
y_validation = validation_data["TX_FRAUD"]
y_test = test_data["TX_FRAUD"]
assert train_data["TX_DATETIME"].max() < validation_data["TX_DATETIME"].min()
assert validation_data["TX_DATETIME"].max() < test_data["TX_DATETIME"].min()
assert len(X_test_behavioral) == 287873 and int(y_test.sum()) == 2547
assert all(np.isfinite(X.to_numpy()).all() for X in [
    X_train_behavioral, X_validation_behavioral, X_test_behavioral
])
print("Split sizes:", len(y_train), len(y_validation), len(y_test))


## Model training and calibration

The pipeline fits the scaler and classifier on training data. It fits isotonic calibration on validation data. The seven model features do not contain previous fraud labels.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, roc_auc_score

behavioral_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ))
])
behavioral_model.fit(X_train_behavioral, y_train)
isotonic_calibrated_model = CalibratedClassifierCV(
    estimator=behavioral_model, method="isotonic", cv="prefit"
)
isotonic_calibrated_model.fit(X_validation_behavioral, y_validation)
original_probabilities = isotonic_calibrated_model.predict_proba(X_test_behavioral)[:, 1]
print("Test average precision:", average_precision_score(y_test, original_probabilities))
print("Test ROC-AUC:", roc_auc_score(y_test, original_probabilities))


## Validation-selected threshold defense

Original threshold: 0.145, from the original study’s validation grid. Defense selection maximizes F1 on 20%-reduced validation data, with a threshold no higher than 0.145 and an attacked-validation alert budget of twice the original clean-validation alerts. The same August labels also fitted calibration. Test labels are not used by this threshold-selection cell.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

required = [
    "behavioral_model", "isotonic_calibrated_model",
    "X_validation_behavioral", "y_validation",
    "X_test_behavioral", "y_test"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing variables: {missing}")

final_folder = results_folder
final_folder.mkdir(exist_ok=True)

original_threshold = 0.145
reductions = [0.00, 0.05, 0.10, 0.15, 0.20]

def modify_fraud_amounts(X, labels, reduction):
    """Modify fraud amounts; preserve history and legitimate transactions."""
    changed = X.copy()
    positions = np.flatnonzero(np.asarray(labels) == 1)

    amounts = X.iloc[positions]["TX_AMOUNT"].to_numpy()
    historical_mean = (
        amounts
        - X.iloc[positions]["CUSTOMER_AMOUNT_DIFF"].to_numpy()
    )

    if not np.all(amounts > 0):
        raise ValueError("Fraud amounts must be positive.")

    changed.iloc[
        positions, X.columns.get_loc("TX_AMOUNT")
    ] = amounts * (1 - reduction)

    changed.iloc[
        positions, X.columns.get_loc("CUSTOMER_AMOUNT_DIFF")
    ] = amounts * (1 - reduction) - historical_mean

    return changed

val_labels = np.asarray(y_validation).astype(int)
test_labels = np.asarray(y_test).astype(int)

val_clean_probs = isotonic_calibrated_model.predict_proba(
    X_validation_behavioral
)[:, 1]

# Defense-selection scenario: 20% reduction on validation fraud.
val_attacked = modify_fraud_amounts(
    X_validation_behavioral, val_labels, 0.20
)
val_attacked_probs = isotonic_calibrated_model.predict_proba(
    val_attacked
)[:, 1]

original_val_alerts = int(
    (val_clean_probs >= original_threshold).sum()
)
alert_budget = min(len(val_labels), 2 * original_val_alerts)

# Compute results at every distinct validation attack score.
# Group tied probabilities together so >= threshold is respected.
order = np.argsort(-val_attacked_probs, kind="stable")
sorted_scores = val_attacked_probs[order]
cumulative_tp = np.cumsum(val_labels[order])

ends = np.r_[
    np.flatnonzero(sorted_scores[:-1] != sorted_scores[1:]),
    len(sorted_scores) - 1
]

candidate_thresholds = sorted_scores[ends]
candidate_alerts = ends + 1
candidate_tp = cumulative_tp[ends]
candidate_fp = candidate_alerts - candidate_tp
candidate_fn = val_labels.sum() - candidate_tp

candidate_f1 = (
    2 * candidate_tp
    / (2 * candidate_tp + candidate_fp + candidate_fn)
)

eligible = (
    (candidate_thresholds <= original_threshold)
    & (candidate_alerts <= alert_budget)
)

if not eligible.any():
    raise RuntimeError("No lower threshold meets the alert budget.")

eligible_indices = np.flatnonzero(eligible)
best_index = eligible_indices[
    np.argmax(candidate_f1[eligible_indices])
]
defense_threshold = float(candidate_thresholds[best_index])

print("Original threshold:", original_threshold)
print("Defense threshold:", repr(defense_threshold))
print("Original clean validation alerts:", original_val_alerts)
print("Validation alert budget:", alert_budget)
print("Selected attacked-validation alerts:",
      int(candidate_alerts[best_index]))
print("Selected attacked-validation F1:",
      round(float(candidate_f1[best_index]), 4))
print("Threshold selection complete — test data was not used.")

## Test attack and defense evaluation

Modify all fraudulent test transactions at each reduction independently. Legitimate transactions remain unchanged. A 0% control checks prediction consistency. Each threshold uses its own clean detected-fraud count as the attack-success denominator.

In [ ]:
test_clean_probs = isotonic_calibrated_model.predict_proba(
    X_test_behavioral
)[:, 1]

test_probabilities = {0.00: test_clean_probs}

for reduction in reductions[1:]:
    attacked_test = modify_fraud_amounts(
        X_test_behavioral, test_labels, reduction
    )
    test_probabilities[reduction] = (
        isotonic_calibrated_model.predict_proba(attacked_test)[:, 1]
    )

# Confirm the 0% transformation preserves predictions.
zero_change = modify_fraud_amounts(
    X_test_behavioral, test_labels, 0.00
)
zero_probs = isotonic_calibrated_model.predict_proba(
    zero_change
)[:, 1]

assert np.allclose(
    zero_probs, test_clean_probs, rtol=0, atol=1e-12
), "0% control failed."

fraud = test_labels == 1
rows = []

for setting, threshold in [
    ("Original", original_threshold),
    ("Defense", defense_threshold)
]:
    clean_detected = test_clean_probs >= threshold
    original_detected_fraud = clean_detected & fraud
    denominator = int(original_detected_fraud.sum())

    for reduction in reductions:
        detected = test_probabilities[reduction] >= threshold

        tp = int((detected & fraud).sum())
        fp = int((detected & ~fraud).sum())
        fn = int((~detected & fraud).sum())
        tn = int((~detected & ~fraud).sum())

        evaded = int((original_detected_fraud & ~detected).sum())
        newly_detected = int(
            (fraud & ~clean_detected & detected).sum()
        )

        rows.append({
            "Setting": setting,
            "Threshold": threshold,
            "Amount_Reduction_%": int(round(reduction * 100)),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "Precision_%": 100 * tp / (tp + fp)
                           if tp + fp else 0,
            "Recall_%": 100 * tp / (tp + fn),
            "F1": 2 * tp / (2 * tp + fp + fn),
            "Alert_Rate_%": 100 * (tp + fp) / len(test_labels),
            "False_Positive_Rate_%": 100 * fp / (fp + tn),
            "Originally_Detected_Fraud": denominator,
            "Evaded": evaded,
            "Newly_Detected": newly_detected,
            "Attack_Success_Rate_%": 100 * evaded / denominator
                                   if denominator else np.nan
        })

final_results = pd.DataFrame(rows)

final_results.to_csv(
    final_folder / "final_attack_defense_comparison.csv",
    index=False
)

display(final_results.round(4))
print("Evaluation saved.")

## Charts and model snapshots

The output path is `results/reproduced`. Version metadata records the packages used. Saved model snapshots are optional local outputs, not prerequisites for running the notebook.

In [ ]:
import matplotlib.pyplot as plt
import joblib
import json
import sklearn

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for setting in ["Original", "Defense"]:
    subset = final_results[
        final_results["Setting"] == setting
    ].sort_values("Amount_Reduction_%")

    for axis, metric, title in [
        (axes[0], "Recall_%", "Fraud recall"),
        (axes[1], "Precision_%", "Alert precision"),
        (axes[2], "Alert_Rate_%", "Total alert rate")
    ]:
        axis.plot(
            subset["Amount_Reduction_%"],
            subset[metric],
            marker="o",
            label=setting
        )
        axis.set_title(title)
        axis.set_xlabel("Fraud amount reduction (%)")
        axis.set_ylabel("Percent")
        axis.set_xticks([0, 5, 10, 15, 20])
        axis.grid(alpha=0.25)
        axis.legend()

fig.suptitle("September test results: original and defense thresholds")
fig.tight_layout()

fig.savefig(
    final_folder / "attack_defense_comparison.png",
    dpi=300, bbox_inches="tight"
)
fig.savefig(
    final_folder / "attack_defense_comparison.pdf",
    bbox_inches="tight"
)
plt.show()

joblib.dump(
    behavioral_model,
    final_folder / "behavioral_model.joblib"
)
joblib.dump(
    isotonic_calibrated_model,
    final_folder / "isotonic_calibrated_model.joblib"
)

metadata = {
    "original_threshold": original_threshold,
    "defense_threshold": defense_threshold,
    "feature_order": X_test_behavioral.columns.tolist(),
    "test_rows": len(test_labels),
    "test_fraud": int(fraud.sum()),
    "amount_reductions": reductions,
    "defense_selection": (
        "Maximum F1 on 20%-amount-reduced validation data; "
        "threshold <= 0.145; alerts <= twice original clean "
        "validation alerts."
    ),
    "validation_alert_budget": int(alert_budget),
    "historical_feature_rule": (
        "Freeze historical averages and counts; recompute "
        "CUSTOMER_AMOUNT_DIFF after changing TX_AMOUNT."
    ),
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__
    }
}

(final_folder / "experiment_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("Charts, models, and experiment settings saved.")

## Interpretation

The supplied run showed 217 of 639 originally detected fraud transactions evading detection under a 20% amount reduction. Recall fell from 25.09% to 16.57%. The lower threshold detected 12 additional fraud transactions and produced 94 additional false alerts. The repository report describes limitations. Inspect regenerated results before making claims that a new run matches these values.